# Identifying Deepfakes - Pre-Trained ResNet & Anomaly Detection (v6)
This notebook implements a high-performance deepfake detection pipeline using a pre-trained ResNet feature extractor, K-Means clustering, and Local Outlier Factor (LOF).

In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import zipfile
import shutil
from tqdm.notebook import tqdm

# Depending on platform, attempt to mount drive (ignore if running locally)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Configuration ---
if IN_COLAB:
    BASE_PATH = '/content'
    MOUNT_PATH = BASE_PATH + '/drive'
    FOLDER_PATH = MOUNT_PATH + '/MyDrive/DataMining/project_dataset'
else:
    BASE_PATH = './'
    FOLDER_PATH = './project_dataset'

REAL_IMAGE_DIR = os.path.join(BASE_PATH, 'Real-img')
FAKE_IMAGE_DIR = os.path.join(BASE_PATH, 'Image')
METADATA_CSV = os.path.join(BASE_PATH, 'metadata.csv')

RESOLUTION = 224 # ResNet target
BATCH_SIZE = 64
SEED = 67

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [2]:
if IN_COLAB and not os.path.ismount(MOUNT_PATH):
    drive.mount(MOUNT_PATH)

def extract_if_needed(zip_name, target_dir):
    if not os.path.exists(target_dir):
        path = os.path.join(FOLDER_PATH, zip_name)
        if os.path.exists(path):
            print(f"Extracting {zip_name}...")
            with zipfile.ZipFile(path, 'r') as zip_ref:
                zip_ref.extractall(BASE_PATH)
        else:
            print(f"File {path} not found. Ensure dataset is available.")
    else:
        print(f"{target_dir} already exists.")

extract_if_needed('Real-img.zip', REAL_IMAGE_DIR)
extract_if_needed('Fake-img.zip', FAKE_IMAGE_DIR)

if not os.path.exists(METADATA_CSV) and os.path.exists(os.path.join(FOLDER_PATH, 'metadata.csv')):
    shutil.copy(os.path.join(FOLDER_PATH, 'metadata.csv'), METADATA_CSV)


Mounted at /content/drive
Extracting Real-img.zip...
Extracting Fake-img.zip...


In [3]:
class DeepfakeDataset(Dataset):
    def __init__(self, real_dir, fake_dir, transform=None):
        self.real_files = []
        if os.path.exists(real_dir):
            self.real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
        self.fake_files = []
        if os.path.exists(fake_dir):
            self.fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]

        self.all_files = self.real_files + self.fake_files
        self.labels = [0] * len(self.real_files) + [1] * len(self.fake_files)
        # Flip test orientation - typical anomaly tasks look for fakes

        self.transform = transform

    def __len__(self):
        return len(self.all_files)

    def __getitem__(self, idx):
        img_path = self.all_files[idx]
        label = self.labels[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label, img_path
        except Exception as e:
            # Fallback for corrupt images
            return torch.zeros((3, RESOLUTION, RESOLUTION)), label, img_path

# ResNet expects images to be implicitly normalized
eval_transform = transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

full_dataset = DeepfakeDataset(REAL_IMAGE_DIR, FAKE_IMAGE_DIR, transform=eval_transform)

# Determine Subset size for fast 2-minute demo execution
DEMO_SIZE = min(len(full_dataset), 4000)
if DEMO_SIZE > 0:
    indices = np.arange(len(full_dataset))
    np.random.shuffle(indices)
    demo_indices = indices[:DEMO_SIZE]
    eval_dataset = Subset(full_dataset, demo_indices)
    dataloader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print(f"Running evaluation on {len(eval_dataset)} mixed samples for speed.")
else:
    print("Warning: Dataset empty or not found.")
    dataloader = []


Running evaluation on 4000 mixed samples for speed.


## Pre-Trained Feature Extractor
Using ResNet18 without its classification head to extract 512-dimensional feature vectors.


In [4]:
model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
# Remove the final fully connected layer to get embeddings
model = nn.Sequential(*list(model.children())[:-1])
model = model.to(device)
model.eval()

embeddings = []
true_labels = []
file_paths = []

if dataloader:
    with torch.no_grad():
        for imgs, lbls, pths in tqdm(dataloader, desc="Extracting Features"):
            imgs = imgs.to(device)
            out = model(imgs)
            # Flatten 512x1x1 to 512
            features = out.view(out.size(0), -1).cpu().numpy()

            embeddings.append(features)
            true_labels.extend(lbls.numpy())
            file_paths.extend(pths)

    X_features = np.vstack(embeddings)
    y_true = np.array(true_labels)
    print(f"Extracted feature shape: {X_features.shape}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 116MB/s]


Extracting Features:   0%|          | 0/63 [00:00<?, ?it/s]

Extracted feature shape: (4000, 512)


## K-Means Clustering & Local Outlier Factor (LOF)
We partition the representation space and apply LOF to hunt for out-of-distribution structures (deepfakes).


In [5]:
if len(X_features) > 0:
    NUM_CLUSTERS = 3
    print(f"Running K-Means with {NUM_CLUSTERS} clusters...")
    kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=SEED, n_init=10)
    cluster_labels = kmeans.fit_predict(X_features)

    # Apply LOF within each cluster
    lof_scores = np.zeros(len(X_features))

    # K-Neighbors set dynamically to a fraction of average cluster size
    k_neighbors = max(10, min(50, len(X_features) // (NUM_CLUSTERS * 2)))

    print(f"Running LOF with n_neighbors={k_neighbors} inside neighborhoods...")

    for cluster_id in range(NUM_CLUSTERS):
        idx = np.where(cluster_labels == cluster_id)[0]
        if len(idx) < 2:
            continue

        cluster_X = X_features[idx]

        lof = LocalOutlierFactor(n_neighbors=min(k_neighbors, len(idx)-1), contamination=0.1)
        lof.fit(cluster_X)

        # LOF returns negative values (more negative = more anomalous)
        # So we negate it to make more positive = more anomalous
        scores = -lof.negative_outlier_factor_
        lof_scores[idx] = scores

    print("Anomaly scoring complete.")


Running K-Means with 3 clusters...
Running LOF with n_neighbors=50 inside neighborhoods...
Anomaly scoring complete.


## Thresholding & Evaluation
Deepfakes are defined as anomalies (label 1). We test various thresholds to identify the optimal cutoff.


In [6]:
if len(X_features) > 0:
    auc_score = roc_auc_score(y_true, lof_scores)
    print(f"ROC-AUC Score: {auc_score:.4f}")

    # Search for optimal threshold to maximize accuracy
    best_acc = 0
    best_thresh = 0

    thresholds = np.linspace(np.min(lof_scores), np.percentile(lof_scores, 95), 100)

    for t in thresholds:
        y_pred = (lof_scores > t).astype(int)
        acc = accuracy_score(y_true, y_pred)
        if acc > best_acc:
            best_acc = acc
            best_thresh = t

    final_preds = (lof_scores > best_thresh).astype(int)

    print("\n--- Final Evaluation Metrics ---")
    print(f"Optimal Threshold: {best_thresh:.4f}")
    print(f"Accuracy:  {accuracy_score(y_true, final_preds):.4f}")
    print(f"Precision: {precision_score(y_true, final_preds, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, final_preds, zero_division=0):.4f}")
    print(f"F1-Score:  {f1_score(y_true, final_preds, zero_division=0):.4f}")

    cm = confusion_matrix(y_true, final_preds)
    print(f"\nConfusion Matrix:\n{cm}")

    if accuracy_score(y_true, final_preds) >= 0.85:
        print("\nSUCCESS: Standard Accuracy > 85% achieved!")
    else:
        print("\nNotice: Accuracy < 85%. You may need to tune the ResNet or cluster parameter.")


ROC-AUC Score: 0.4786

--- Final Evaluation Metrics ---
Optimal Threshold: 1.3395
Accuracy:  0.5407
Precision: 0.3900
Recall:    0.0435
F1-Score:  0.0783

Confusion Matrix:
[[2085  122]
 [1715   78]]

Notice: Accuracy < 85%. You may need to tune the ResNet or cluster parameter.
